# Clean Operator Data Reports
Code author: Audrey McManemin

Edited from code written by Sahar H. El Abbadi

Date started: 2024-09-09
Date last edited: 2025-02-20


### Notes
- All raw report inputs are saved in 00_raw_reports. No changes are manually made to operator reports (except for date propagation in release schedule). All cleaning is handled in Python.


In [1]:
# Imports
import pandas as pd
import numpy as np
import pathlib
from methods_clean_operator_reports import operator_qc, clean_aeromon, clean_gsma, clean_seekops, clean_uhei, clean_dtu, clean_flylogix, clean_empa, clean_uu_lsce_cyi_rhul

# Load and clean raw data submitted by operators
### Notes on formatting:
- Operators added their own QC indicators, thus not all columns are uniform across reports
- Values left in the Excel file are replaced during import into PyCharm with "nan"
- Naming convention for dataframes: operator

## Notes on Cleaning Operator Data

### Generate data frame with the following columns:
- Operator: name of operator (Aeromon, SeekOps, GSMA)
- Week: week that the operator participated in (1, 2, 3, 4)
- ReleaseID: release number for that week, corresponding to the master schedule of all the releases (not just for which that operator measured)
- DateOfSurvey: date in YYYY-MM-DD format
- SurveyStartTime: start time of survey in local time (UTC+2)
- SurveyEndTime: end time of survey in local time (UTC+2)
- QuantifiedPlume: boolean input, 1 indicates operator submitted a valid quantification estimate for this overpass (excludes quantification estimates that are provided but fail operator QC standards)
- EstimatedEmissionRate: estimated emissions in kgh
- EstimatedEmissionRateUpper: upper bound of uncertainty on quantification estimate
- EstimatedEmissionRateLower: lower bound of uncertainty on quantification estimate
- UncertaintyType: type of uncertainty for upper and lower values reported above
- OperatorWindspeed: operator reported windspeed in m/s
- QCFlag: operator specific QC flag
- OperatorKept: operator submitted estimates for this result
- EstimateType: if applicable, the method used for this estimate

### Aeromon
#### Submission details
- Submitted on 2024-07-22
- Resubmitted results & schedule on 2024-08-13 after confirmation that they would not receive wind data from the TADI facility 

### Required data cleaning
- windspeed parsing

In [2]:
# import data
aeromon_results_path = pathlib.PurePath('00_raw_reports', 'Aeromon_results_submitted_08-13.xlsx')
aeromon_schedule_path = pathlib.PurePath('00_raw_reports', 'Aeromon_release_schedule.xlsx')

aeromon_results = pd.read_excel(aeromon_results_path, sheet_name='Reported Data', engine='openpyxl')
aeromon_schedule = pd.read_excel(aeromon_schedule_path, engine='openpyxl', skiprows=1, usecols='D:J')

# clean data
aeromon_clean = clean_aeromon(aeromon_schedule, aeromon_results)

# save data
aeromon_clean.to_csv(pathlib.PurePath('01_clean_reports', 'aeromon_clean.csv'))


### GMSA
#### Submission details
- Submitted on 2024-07-22

#### Required data cleaning
- Uncertainty type parsing

In [2]:
# import data
gsma_results_path = pathlib.PurePath('00_raw_reports', 'GSMA_results_submitted_07-22.xlsx')
gsma_schedule_path = pathlib.PurePath('00_raw_reports', 'GSMA_release_schedule.xlsx')

gsma_results = pd.read_excel(gsma_results_path, sheet_name='Reported Data', engine='openpyxl')
gsma_schedule = pd.read_excel(gsma_schedule_path, engine='openpyxl', skiprows=1, usecols='D:J')

# clean data
gsma_clean = clean_gsma(gsma_schedule, gsma_results)

# save data
gsma_clean.to_csv(pathlib.PurePath('01_clean_reports', 'gsma_clean.csv'))

### SeekOps
#### Submission details
- Submitted on 2024-07-29

#### Required data cleaning
- SeekOps reported results for all flights that they took, even if 2 flights were during the same release. They reported an totaled estimate for those flights in the schedule but the results section contains information per flight, not per release. 
- They report a standard 30% uncertainty
- SeekOps did not report upper and lower bounds of emission estimates
- Because SeekOps reported the flight start and end time, and could have conducted many flights during one survey, the release start and end time is used here. 

In [2]:
# import data
seekops_results_path = pathlib.PurePath('00_raw_reports', 'SeekOps_results_submitted_07-29.xlsx')
seekops_schedule_path = pathlib.PurePath('00_raw_reports', 'SeekOps_release_schedule.xlsx')

seekops_results = pd.read_excel(seekops_results_path, sheet_name='Reported Data', engine='openpyxl')
seekops_schedule = pd.read_excel(seekops_schedule_path, engine='openpyxl', usecols='D:J')

# clean data
seekops_clean = clean_seekops(seekops_schedule, seekops_results)

# save data
seekops_clean.to_csv(pathlib.PurePath('01_clean_reports', 'seekops_clean.csv'))

### Heidelberg University (UHEI)
#### Submission details
- Submitted results on 2024-08-09
- Submitted schedule on 2024-09-02
- Participant in Week 2

#### Required data cleaning
- UHEI used different methods (Gaussian plume model and OTM-33) during their analysis. This is marked in the EstimateType column
- Included all releases in their results, including ones that they did not measure / they failed to measure.  
- Convert start and end time to local time (UTC+2) from UTC
- Upper and lower emission rates were in the format "(emission rate)+/-(delta)"

In [2]:
# import data
uhei_results_path = pathlib.PurePath('00_raw_reports', 'UHEI_results_submitted_08-09.xlsx')
uhei_schedule_path = pathlib.PurePath('00_raw_reports', 'UHEI_release_schedule.xlsx')

uhei_results = pd.read_excel(uhei_results_path, sheet_name='Reported Data', engine='openpyxl')
uhei_schedule = pd.read_excel(uhei_schedule_path, engine='openpyxl', skiprows=1, usecols='D:J')

# clean data
uhei_clean = clean_uhei(uhei_schedule, uhei_results)

# save data
uhei_clean.to_csv(pathlib.PurePath('01_clean_reports', 'uhei_clean.csv'))


### Technical University of Denmark (DTU)
#### Submission details
- Submitted first results on 2024-09-12 but not in the template required
- Resubmitted results in the format requested on 2024-09-20
- Participant in Week 1

#### Required data cleaning
- Did not turn in a schedule as requested - however all the information is in the results spreadsheet and they drove during every release
- They turned in an estimate for all releases except for ones which they designated as BDL (below detection limit). I have included those here as zeros

In [3]:
# import data
dtu_results_path = pathlib.PurePath('00_raw_reports', 'DTU_results_submitted_09-20.xlsx')

dtu_results = pd.read_excel(dtu_results_path, sheet_name='Reported Data', engine='openpyxl')

# clean data
dtu_clean = clean_dtu(dtu_results)

# save data
dtu_clean.to_csv(pathlib.PurePath('01_clean_reports', 'dtu_clean.csv'))

## Flylogix

Submission details
- Submitted results on 2024-10-21 
- Submitted schedule later that same day 
- Participant in Week 4

Required data cleaning

In [2]:
# import data
flylogix_results_path = pathlib.PurePath('00_raw_reports', 'Flylogix_results_submitted_10_21.xlsx')
flylogix_schedule_path = pathlib.PurePath('00_raw_reports', 'Flylogix_release_schedule.xlsx')
flylogix_results = pd.read_excel(flylogix_results_path, sheet_name='Reported Data', engine='openpyxl')
flylogix_schedule = pd.read_excel(flylogix_schedule_path, engine='openpyxl', skiprows=1, usecols='D:J')

# clean data
flylogix_clean = clean_flylogix(flylogix_schedule, flylogix_results)

# save data
flylogix_clean.to_csv(pathlib.PurePath('01_clean_reports', 'flylogix_clean.csv'))

## EMPA
Submission details 
- Submitted results and schedule on 2024-11-29
- Participant in Week 4

Required data cleaning
- Submitted average of overpasses in tab Reported Data Avg (and also individual overpasses in Reported Data sheet)

In [2]:
# EMPA data cleaning
# import data
empa_results_path = pathlib.PurePath('00_raw_reports', 'EMPA_results_submitted_11-29.xlsx')
empa_schedule_path = pathlib.PurePath('00_raw_reports', 'EMPA_release_schedule.xlsx')
empa_results = pd.read_excel(empa_results_path, sheet_name='Reported Data Avg', engine='openpyxl')
empa_schedule = pd.read_excel(empa_schedule_path, engine='openpyxl', skiprows=1, usecols='D:J')

# clean data
empa_clean = clean_empa(empa_schedule, empa_results)

# save data
empa_clean.to_csv(pathlib.PurePath('01_clean_reports', 'empa_clean.csv'))

## UU/LSCE/CYI/RHUL Team

Submission details
- Participant in Week 4
- Submitted results on 2024-12-02 and schedule when prompted on 2024-12-03

Required data cleaning
- Kept all releases in results, even ones they did not measure

In [2]:
# UU/LSCE/CYI/RHUL data cleaning

# import data
uu_lsce_cyi_rhul_results_path = pathlib.PurePath('00_raw_reports', 'UU_LSCE_CYI_RHUL_results_submitted_12-02.xlsx')
uu_lsce_cyi_rhul_schedule_path = pathlib.PurePath('00_raw_reports', 'UU_LSCE_CYI_RHUL_release_schedule.xlsx')
uu_lsce_cyi_rhul_results = pd.read_excel(uu_lsce_cyi_rhul_results_path, sheet_name='Reported Data', engine='openpyxl')
uu_lsce_cyi_rhul_schedule = pd.read_excel(uu_lsce_cyi_rhul_schedule_path, engine='openpyxl', skiprows=1, usecols='D:J')

# clean data
uu_lsce_cyi_rhul_clean = clean_uu_lsce_cyi_rhul(uu_lsce_cyi_rhul_schedule, uu_lsce_cyi_rhul_results)

# save data
uu_lsce_cyi_rhul_clean.to_csv(pathlib.PurePath('01_clean_reports', 'uu_lsce_cyi_rhul_clean.csv'))